# Quickstart: is the best backtest skill or luck?

We try 200 trading strategies that have **no skill at all** and look at the best one, first
the way a backtest report would, then with `overfit`.

In [1]:
import numpy as np
import pandas as pd

import overfit

rng = np.random.default_rng(42)
n_days, n_strategies = 5 * 252, 200
daily_vol = 0.15 / np.sqrt(252)
market = rng.standard_normal((n_days, 1))
noise = rng.standard_normal((n_days, n_strategies))
returns = pd.DataFrame(
    daily_vol * (np.sqrt(0.3) * market + np.sqrt(0.7) * noise),
    index=pd.bdate_range("2020-01-01", periods=n_days),
    columns=[f"s{i:03d}" for i in range(n_strategies)],
)
sharpe = overfit.sharpe_ratio(returns, periods_per_year=252)
best = sharpe.idxmax()
stats = overfit.describe(returns[best], periods_per_year=252)
print(
    f"best strategy {best}: annual Sharpe {stats.sr_annual:.2f}, "
    f"probabilistic Sharpe ratio {stats.psr:.3f}"
)

best strategy s045: annual Sharpe 1.02, probabilistic Sharpe ratio 0.989


On its own the best strategy looks convincing. But it is the maximum of 200 tries. The
**deflated Sharpe ratio** measures it against the Sharpe ratio we would expect from the best
of 200 strategies with no skill:

In [2]:
dsr = overfit.deflated_sharpe_from_trials(returns, periods_per_year=252)
print(f"expected best Sharpe with no skill: {dsr.sr0_annual:.2f}")
print(f"deflated Sharpe ratio: {dsr.dsr:.3f}   (needs >= 0.95)")

expected best Sharpe with no skill: 1.09
deflated Sharpe ratio: 0.440   (needs >= 0.95)


The bootstrap tests ask the same question without distributional assumptions and keep the
correlation between strategies:

In [3]:
spa = overfit.spa_test(returns, criterion="sharpe", seed=0)
rw = overfit.romano_wolf(returns, criterion="sharpe", seed=0)
pbo = overfit.pbo(returns)
print(f"Hansen SPA p-value: {spa.pvalue:.3f}")
print(f"strategies significant after Romano-Wolf: {rw.n_rejected}")
print(f"probability of backtest overfitting: {pbo.pbo:.2f}")

Hansen SPA p-value: 0.509
strategies significant after Romano-Wolf: 0
probability of backtest overfitting: 0.34


`overfit.audit` runs every check at once and writes a report:

In [4]:
from IPython.display import Markdown

report = overfit.audit(returns, seed=0)
Markdown(report.to_markdown())

# Backtest audit

200 strategies, 1260 observations (5.0 years), criterion: sharpe, benchmark: zero.

| Check | Result |
|---|---|
| Best strategy (in-sample) | `s045` |
| Annualised Sharpe ratio | 1.02 (HAC s.e. 0.48) |
| Naive one-sided p-value | 0.0172 |
| Trials (effective) | 200 (142.0) |
| Expected best Sharpe under no skill | 1.09 |
| Minimum backtest length for this Sharpe and the effective trials | 6.7 years (sample: 5.0 years) |
| Deflated Sharpe ratio (pass if >= 0.95) | 0.440 (0.480 with effective trials) |
| Haircut Sharpe ratio (Bonferroni) | 0.00 (100% haircut) |
| White Reality Check p-value | 0.512 |
| Hansen SPA p-value (consistent) | 0.501 |
| Strategies significant after Romano-Wolf (5% FWER) | 0 |
| Probability of backtest overfitting (CSCV) | 0.344 |
| P(in-sample winner loses out of sample) | 0.397 |

**Verdict:** Significant on its own, but not after accounting for the trials: consistent with selection luck.

Bootstrap: stationary, mean block length 1.6, 2000 replications.


## A strategy that does have an edge

Give one strategy a true annual Sharpe ratio of about 2.5 and the verdict changes:

In [5]:
skilled = returns.copy()
skilled["s007"] += 2.5 * daily_vol / np.sqrt(252)
report = overfit.audit(skilled, seed=0)
print(report.best, "|", report.verdict)
print(
    f"DSR {report.dsr:.3f}, SPA p-value {report.pvalue_spa:.3f}, "
    f"Romano-Wolf rejections {report.n_romano_wolf}"
)

s007 | Evidence of skill survives the multiple-testing corrections.
DSR 0.999, SPA p-value 0.000, Romano-Wolf rejections 1


## Real data: 1,244 moving-average and momentum rules on the S&P 500

The index data ship with the `arch` package (1999-2018). Each rule is long or flat, trades
with a one-day lag and pays 5 basis points per trade; the benchmark is buy-and-hold.

In [6]:
from arch.data import sp500

from overfit.rules import build_rules

prices = sp500.load()["Adj Close"]
rules = build_rules(prices, families=["ma", "momentum"])
rule_returns = rules.returns(cost=0.0005)
buy_and_hold = rules.asset_returns()
report = overfit.audit(rule_returns, buy_and_hold, seed=0)
Markdown(report.to_markdown())

# Backtest audit

1244 strategies, 4770 observations (18.9 years), criterion: sharpe, benchmark: supplied series.

| Check | Result |
|---|---|
| Best strategy (in-sample) | `ma:s=25,l=200,b=0.01,d=0` |
| Annualised Sharpe ratio of returns in excess of the benchmark | 0.15 (HAC s.e. 0.19) |
| Naive one-sided p-value | 0.2242 |
| Trials (effective) | 1244 (86.0) |
| Expected best Sharpe under no skill | 0.52 |
| Minimum backtest length for this Sharpe and the effective trials | 285.1 years (sample: 18.9 years) |
| Deflated Sharpe ratio (pass if >= 0.95) | 0.054 (0.149 with effective trials) |
| Haircut Sharpe ratio (Bonferroni) | 0.00 (100% haircut) |
| White Reality Check p-value | 0.552 |
| Hansen SPA p-value (consistent) | 0.528 |
| Strategies significant after Romano-Wolf (5% FWER) | 0 |
| Probability of backtest overfitting (CSCV) | 0.288 |
| P(in-sample winner loses out of sample) | 0.592 |

**Verdict:** No evidence of skill, even before correcting for the number of trials.

Bootstrap: stationary, mean block length 17.5, 2000 replications.


## Cross-validation with overlapping labels

When each label looks 20 days ahead, neighbouring samples share most of their label.
`PurgedKFold` drops those neighbours from training; it plugs into scikit-learn like any
other splitter.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score

r = prices.pct_change()
X = pd.concat({f"ret_{n}": prices.pct_change(n) for n in (1, 5, 20, 60)}, axis=1)
X["vol_20"] = r.rolling(20).std()
y = (prices.shift(-20) / prices - 1 > 0).astype(int)
keep = X.notna().all(axis=1) & prices.shift(-20).notna()
X, y = X[keep], y[keep]
t1 = overfit.label_times(len(X), horizon=20)

model = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=0)
for name, cv in {
    "shuffled k-fold": KFold(5, shuffle=True, random_state=0),
    "purged k-fold": overfit.PurgedKFold(5, t1=t1, embargo=0.01),
}.items():
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()
    print(f"{name:>16}: AUC {auc:.3f}")

 shuffled k-fold: AUC 0.666


   purged k-fold: AUC 0.480
